In [30]:
import pandas as pd


# Loaded variable 'df' from URI: c:\Users\barba\Documents\interview\Sales_Transactions_Interview_Dataset.csv
column_types = {
    "TransactionNo": "string",
    "Date": "string",
    "ProductNo": "string",
    "ProductName": "string",
    "Price": "float64",
    "Quantity": "Int64",
    "CustomerNo": "Int64",
    "Country": "string"
}
date_columns = list(["Date"])
df = pd.read_csv(r'c:\Users\barba\Documents\interview\Sales_Transactions_Interview_Dataset.csv')
original_df=df.copy()
df = df.dropna()

for col, dtype in column_types.items():
    if dtype  in ["Int64", "int64", "float64"]: 
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].astype(dtype)
df = df.dropna()

for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce")
na_kpi_count=original_df.shape[0]-df.shape[0]
for col in df.columns:
    if df[col].dtype == "object" and col not in ["Date", "Country",]:
        df[col] = df[col].str.strip()
mask = df[["Price", "Quantity"]].gt(0).all(axis=1)
df = df[mask].copy()
below_or_0_kpi_count = df[~mask].shape[0]
df['gross_revenue']=df['Price']*df['Quantity']
Sales_per_country = (
    df.groupby(["Date", "Country"], as_index=False)
      .agg({
          "gross_revenue": ["sum", "mean"],
          "TransactionNo": ["count"]
      })
)
Sales_per_country[("gross_revenue", "sum")] = Sales_per_country[("gross_revenue", "sum")].round(3)
Sales_per_country[("gross_revenue", "mean")] = Sales_per_country[("gross_revenue", "mean")].round(3)

C:\Users\barba\AppData\Local\Temp\ipykernel_21756\3765567457.py:34: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  below_or_0_kpi_count = df[~mask].shape[0]


In [31]:
(~mask).sum()

np.int64(8531)

In [32]:
import numpy as np
import pandas as pd

from bokeh.io import output_notebook, show
from bokeh.models import ColumnDataSource, Select, CustomJS, HoverTool, DatetimeTickFormatter
from bokeh.layouts import column
from bokeh.plotting import figure

output_notebook()
daily_revenue = (
    df.groupby(["Date", "Country"], as_index=False)["gross_revenue"]
        .sum()
        .sort_values(["Country", "Date"])
)
def make_daily_revenue_chart(df):
    df = df.copy()

    # Basic cleanup
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce").dt.normalize()
    df["gross_revenue"] = pd.to_numeric(df["gross_revenue"], errors="coerce")

    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["Date", "Country", "gross_revenue"])

    daily_revenue = (
        df.groupby(["Date", "Country"], as_index=False)["gross_revenue"]
          .sum()
          .sort_values(["Country", "Date"])
    )
    
    countries = sorted(daily_revenue["Country"].dropna().unique())
    if not countries:
        raise ValueError("No valid countries found after cleaning.")

    default_country = "United Kingdom" if "United Kingdom" in countries else countries[0]

    data_by_country = {}
    for country in countries:
        group = daily_revenue[daily_revenue["Country"] == country].sort_values("Date")
        data_by_country[country] = {
            "Date": group["Date"].tolist(),
            "gross_revenue": group["gross_revenue"].tolist(),
        }

    source = ColumnDataSource(data=data_by_country[default_country])

    select = Select(
        title="Select Country",
        value=default_country,
        options=list(countries),
    )

    p = figure(
        title=f"Daily Gross Revenue — {default_country}",
        x_axis_type="datetime",
        width=1000,
        height=450,
        toolbar_location="above",
    )

    p.line("Date", "gross_revenue", source=source, line_width=2, legend_label="Revenue")
    p.circle("Date", "gross_revenue", source=source, size=4, alpha=0.7)

    p.xaxis.axis_label = "Date"
    p.yaxis.axis_label = "Gross Revenue"
    p.xaxis.formatter = DatetimeTickFormatter(days="%Y-%m-%d", months="%Y-%m-%d", years="%Y-%m-%d")

    p.add_tools(
        HoverTool(
            tooltips=[
                ("Date", "@Date{%F}"),
                ("Revenue", "@gross_revenue{0,0.00}"),
            ],
            formatters={"@Date": "datetime"},
            mode="vline",
        )
    )

    p.legend.location = "top_left"

    callback = CustomJS(
        args=dict(source=source, data_by_country=data_by_country, plot=p),
        code="""
            const selected = cb_obj.value;
            source.data = data_by_country[selected];
            source.change.emit();
            plot.title.text = `Daily Gross Revenue — ${selected}`;
        """,
    )

    select.js_on_change("value", callback)

    return column(select, p)

show(make_daily_revenue_chart(df))


Loading BokehJS ...

In [33]:
daily_revenue["Date"] = pd.to_datetime(daily_revenue["Date"]).dt.normalize()

dates_with_multiple_countries = (
    daily_revenue.groupby("Date")["Country"]
      .nunique()
      .gt(1)
)

intersection_dates = dates_with_multiple_countries[
    dates_with_multiple_countries
].index

df_intersection = daily_revenue.loc[daily_revenue["Date"].isin(intersection_dates)]
df_intersection

,Date,Country,gross_revenue
0,2018-12-01,Australia,1463.83
31,2018-12-08,Australia,2458.9
72,2018-12-17,Australia,1801.46
106,2019-01-06,Australia,56561.38
119,2019-01-10,Australia,1067.52
...,...,...,...
801,2019-07-08,Unspecified,5239.0
823,2019-07-14,Unspecified,3036.08
881,2019-07-28,Unspecified,3799.02
982,2019-08-22,Unspecified,2440.28


In [34]:
from IPython.display import display

import plotly.express as px

# Create day column from transaction date
df["_transaction_day"] = pd.to_datetime(df["Date"]).dt.date

# Group by customer + day + country and count transactions
transaction_count_distribution = (
    df.groupby(["CustomerNo", "_transaction_day", "Country"])
    .size()
    .reset_index(name="transaction_count")
)

# Histogram
fig = px.histogram(
    transaction_count_distribution,
    x="transaction_count",
    nbins=50,
    title="Distribution of Transactions per Customer-Day-Country",
    labels={
        "transaction_count": "Number of Transactions",
        "count": "Number of Customer-Day-Country Groups"
    }
)

fig.show()

# Optional: show the grouped data
display(
    transaction_count_distribution.sort_values(
        "transaction_count",
        ascending=False
    )
)


,CustomerNo,_transaction_day,Country,transaction_count
7098,14585,2019-10-31,United Kingdom,1111
11995,16219,2019-12-08,United Kingdom,747
9899,15492,2019-12-09,United Kingdom,730
13425,16729,2019-12-05,United Kingdom,720
9864,15475,2019-06-29,United Kingdom,704
...,...,...,...,...
12,12057,2019-11-08,United Kingdom,1
78,12346,2019-01-18,United Kingdom,1
54,12242,2019-06-27,United Kingdom,1
58,12252,2018-12-06,United Kingdom,1


In [35]:
def remove_transaction_count_outliers(
    df,
    date_col,
    country_col,
    customer_id_col,
    quantity_col="Quantity",
    method="iqr",
    iqr_multiplier=1.5,
    percentile_cutoff=0.99,
):
    df = df.copy()

    # Use date only, not full timestamp
    df["_transaction_day"] = pd.to_datetime(df[date_col], errors="coerce").dt.date
    df[quantity_col] = pd.to_numeric(df[quantity_col], errors="coerce")

    group_cols = [customer_id_col, "_transaction_day", country_col]

    # Count transactions for same customer, same day, same country
    transaction_counts = (
        df.groupby(group_cols)
        .size()
        .reset_index(name="transactions_same_customer_day_country")
    )

    def calc_upper_bound(series):
        if method == "iqr":
            q1 = series.quantile(0.25)
            q3 = series.quantile(0.75)
            iqr = q3 - q1
            return q3 + iqr_multiplier * iqr
        elif method == "percentile":
            return series.quantile(percentile_cutoff)
        else:
            raise ValueError("method must be either 'iqr' or 'percentile'")

    transaction_upper_bound = calc_upper_bound(
        transaction_counts["transactions_same_customer_day_country"]
    )
    quantity_upper_bound = calc_upper_bound(df[quantity_col].dropna())

    transaction_counts["is_transaction_count_outlier"] = (
        transaction_counts["transactions_same_customer_day_country"] > transaction_upper_bound
    )

    # Add the count back to original transaction rows
    df_with_counts = df.merge(transaction_counts, on=group_cols, how="left")

    # Quantity outlier at the row level
    df_with_counts["is_quantity_outlier"] = df_with_counts[quantity_col] > quantity_upper_bound

    outlier_mask = (
        df_with_counts["is_transaction_count_outlier"]
        | df_with_counts["is_quantity_outlier"]
    )

    outlier_rows = df_with_counts[outlier_mask].copy()
    cleaned_df = df_with_counts[~outlier_mask].copy()

    return cleaned_df, outlier_rows, {
        "transaction_count_upper_bound": transaction_upper_bound,
        "quantity_upper_bound": quantity_upper_bound,
    }

In [36]:
cleaned_df, outlier_rows, upper_bound = remove_transaction_count_outliers(
    df=df,
    date_col="Date",
    country_col="Country",
    customer_id_col="CustomerNo",
    method="iqr",
    iqr_multiplier=2
)
cleaned_df

,TransactionNo,Date,ProductNo,ProductName,Price,Quantity,CustomerNo,Country,gross_revenue,_transaction_day,transactions_same_customer_day_country,is_transaction_count_outlier,is_quantity_outlier
0,581482,2019-12-09,22485,Set Of 2 Wooden Market Crates,21.47,12,17490,United Kingdom,257.64,2019-12-09,22,False,False
2,581475,2019-12-09,23235,Storage Tin Vintage Leaf,11.53,12,13069,United Kingdom,138.36,2019-12-09,19,False,False
3,581475,2019-12-09,23272,Tree T-Light Holder Willie Winkie,10.65,12,13069,United Kingdom,127.8,2019-12-09,19,False,False
4,581475,2019-12-09,23239,Set Of 4 Knick Knack Tins Poppies,11.94,6,13069,United Kingdom,71.64,2019-12-09,19,False,False
5,581475,2019-12-09,21705,Bag 500g Swirly Marbles,10.65,24,13069,United Kingdom,255.6,2019-12-09,19,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
527759,536585,2018-12-01,37449,Ceramic Cake Stand + Hanging Cakes,20.45,2,17460,United Kingdom,40.9,2018-12-01,1,False,False
527760,536590,2018-12-01,22776,Sweetheart 3 Tier Cake Stand,20.45,1,13065,United Kingdom,20.45,2018-12-01,14,False,False
527761,536590,2018-12-01,22622,Box Of Vintage Alphabet Blocks,20.45,2,13065,United Kingdom,40.9,2018-12-01,14,False,False
527762,536591,2018-12-01,37449,Ceramic Cake Stand + Hanging Cakes,20.45,1,14606,United Kingdom,20.45,2018-12-01,40,False,False


In [37]:
show(make_daily_revenue_chart(cleaned_df))

In [40]:
df["year"] = df["Date"].dt.year
df["month_number"] = df["Date"].dt.month
df["month"] = df["Date"].dt.to_period("M").dt.to_timestamp().dt.strftime("%Y-%m-%d")
df["week"] = (df["Date"] - pd.to_timedelta(df["Date"].dt.weekday, unit="D")).dt.strftime("%Y-%m-%d")
df["day"] = df["Date"].dt.date
df["weekday"] = df["Date"].dt.day_name()
df['gross_revenue']=df['Price']*df['Quantity']

unique_customer_month = (
  df.groupby([ "month", "Country" ], as_index=False)
    .agg(unique_customers=("CustomerNo","nunique"))
    .sort_values(["month","Country"])
)
unique_customer_month

In [53]:
# Create a per-month+country list of unique customers
# Group by month and Country, then aggregate unique CustomerNo values into a list
customer_month = (
    df.groupby(["month", "Country"], as_index=False)
      .agg(list_customer=("CustomerNo", lambda x: list(x.dropna().unique())))
)

customer_month

,month,Country,list_customer
0,2018-12-01,Australia,"[12431, 12386]"
1,2018-12-01,Belgium,"[12383, 12423, 12417, 12395, 12876]"
2,2018-12-01,Cyprus,[12370]
3,2018-12-01,Denmark,[12429]
4,2018-12-01,EIRE,"[14911, 14016, 13337, 13124, 12829, 14156]"
...,...,...,...
286,2019-12-01,Portugal,"[12783, 12766, 12782, 12762]"
287,2019-12-01,Spain,"[17097, 12442]"
288,2019-12-01,Sweden,[17404]
289,2019-12-01,USA,"[12646, 12558]"


In [54]:
# Compute monthly new vs returning customers per country
# Derive month_period from Date to avoid relying on string month column
df['month_period'] = df['Date'].dt.to_period('M')

# Compute first purchase month per customer per country
first_month = (
    df.groupby(['CustomerNo', 'Country'], as_index=False)['Date']
      .min()
)
first_month['first_month'] = pd.to_datetime(first_month['Date']).dt.to_period('M')
first_month = first_month.drop(columns=['Date'])

# Merge first_month back into main df
df = df.merge(first_month, on=['CustomerNo', 'Country'], how='left')

# Keep one row per customer-month-country so each customer counts once per month
monthly_customers = df.drop_duplicates(['CustomerNo', 'Country', 'month_period'])

# Flag whether the customer's first purchase month equals the current month (new customer)
monthly_customers['is_new'] = monthly_customers['first_month'] == monthly_customers['month_period']

# Aggregate counts per month + country
monthly_summary = (
    monthly_customers.groupby(['month_period', 'Country'], as_index=False)
      .agg(
          total_customers=('CustomerNo', 'nunique'),
          new_customers=('is_new', 'sum')
      )
)

monthly_summary['old_customers'] = monthly_summary['total_customers'] - monthly_summary['new_customers']

# Sort and format month for display
monthly_summary = monthly_summary.sort_values(['month_period', 'Country'])
monthly_summary['month'] = monthly_summary['month_period'].astype(str)

# Show final table with month, country, and counts
monthly_summary[['month', 'Country', 'total_customers', 'new_customers', 'old_customers']]


,month,Country,total_customers,new_customers,old_customers
0,2018-12,Australia,2,2,0
1,2018-12,Belgium,5,5,0
2,2018-12,Cyprus,1,1,0
3,2018-12,Denmark,1,1,0
4,2018-12,EIRE,6,6,0
...,...,...,...,...,...
286,2019-12,Portugal,4,0,4
287,2019-12,Spain,2,1,1
288,2019-12,Sweden,1,0,1
289,2019-12,USA,2,1,1


## Buyer KPIs

This section classifies customers using unique order count (`TransactionNo`) within each country:

- `One-time buyer`: exactly 1 unique order in the observed data
- `Repeat buyer`: more than 1 unique order in the observed data

Customer lifetime is measured as **observed lifetime** in this dataset, from first purchase date to last purchase date.

In [55]:
from IPython.display import display
import numpy as np

kpi_source_df = cleaned_df.copy() if "cleaned_df" in globals() else df.copy()

kpi_source_df["Date"] = pd.to_datetime(kpi_source_df["Date"], errors="coerce")
kpi_source_df = kpi_source_df.dropna(subset=["CustomerNo", "TransactionNo", "Country", "Date"]).copy()
kpi_source_df["order_date"] = kpi_source_df["Date"].dt.normalize()

if "gross_revenue" not in kpi_source_df.columns:
    kpi_source_df["gross_revenue"] = kpi_source_df["Price"] * kpi_source_df["Quantity"]

customer_kpis = (
    kpi_source_df.groupby(["CustomerNo", "Country"], as_index=False)
    .agg(
        first_purchase=("order_date", "min"),
        last_purchase=("order_date", "max"),
        order_count=("TransactionNo", "nunique"),
        total_revenue=("gross_revenue", "sum"),
        total_units=("Quantity", "sum"),
    )
)

customer_kpis["buyer_type"] = np.where(
    customer_kpis["order_count"] == 1,
    "One-time buyer",
    "Repeat buyer",
)

buyer_summary = (
    customer_kpis.groupby("buyer_type", as_index=False)
    .agg(
        customers=("CustomerNo", "nunique"),
        total_revenue=("total_revenue", "sum"),
        avg_revenue_per_customer=("total_revenue", "mean"),
        avg_orders_per_customer=("order_count", "mean"),
    )
)

buyer_summary["customer_share_pct"] = (
    buyer_summary["customers"] / buyer_summary["customers"].sum() * 100
).round(2)
buyer_summary["total_revenue"] = buyer_summary["total_revenue"].round(2)
buyer_summary["avg_revenue_per_customer"] = buyer_summary["avg_revenue_per_customer"].round(2)
buyer_summary["avg_orders_per_customer"] = buyer_summary["avg_orders_per_customer"].round(2)

buyer_summary_by_country = (
    customer_kpis.groupby(["Country", "buyer_type"], as_index=False)
    .agg(
        customers=("CustomerNo", "nunique"),
        total_revenue=("total_revenue", "sum"),
        avg_revenue_per_customer=("total_revenue", "mean"),
    )
)

country_totals = buyer_summary_by_country.groupby("Country")["customers"].transform("sum")
buyer_summary_by_country["customer_share_pct"] = (
    buyer_summary_by_country["customers"] / country_totals * 100
).round(2)
buyer_summary_by_country["total_revenue"] = buyer_summary_by_country["total_revenue"].round(2)
buyer_summary_by_country["avg_revenue_per_customer"] = buyer_summary_by_country["avg_revenue_per_customer"].round(2)

repeat_buyer_rate = round((customer_kpis["order_count"] > 1).mean() * 100, 2)

print(f"Repeat buyer rate: {repeat_buyer_rate}%")
display(buyer_summary.sort_values("customers", ascending=False))
display(buyer_summary_by_country.sort_values(["Country", "customers"], ascending=[True, False]))

Repeat buyer rate: 64.58%


,buyer_type,customers,total_revenue,avg_revenue_per_customer,avg_orders_per_customer,customer_share_pct
1,Repeat buyer,2882,25106776.29,8711.58,5.47,64.58
0,One-time buyer,1581,2419647.24,1530.45,1.00,35.42


,Country,buyer_type,customers,total_revenue,avg_revenue_per_customer,customer_share_pct
1,Australia,Repeat buyer,8,69337.45,8667.18,88.89
0,Australia,One-time buyer,1,1184.5,1184.5,11.11
2,Austria,One-time buyer,3,11148.26,3716.09,50.00
3,Austria,Repeat buyer,3,25885.18,8628.39,50.00
4,Bahrain,One-time buyer,2,1428.18,714.09,100.00
...,...,...,...,...,...,...
56,United Arab Emirates,One-time buyer,2,9602.86,4801.43,100.00
58,United Kingdom,Repeat buyer,2618,21696317.91,8287.36,64.40
57,United Kingdom,One-time buyer,1447,2095231.18,1447.98,35.60
60,Unspecified,Repeat buyer,4,20253.47,5063.37,66.67


In [56]:
customer_kpis["customer_lifetime_days"] = (
    customer_kpis["last_purchase"] - customer_kpis["first_purchase"]
).dt.days
customer_kpis["avg_revenue_per_order"] = (
    customer_kpis["total_revenue"] / customer_kpis["order_count"]
).round(2)

customer_lifetime_summary = pd.DataFrame(
    {
        "metric": [
            "Customers",
            "Average observed lifetime (days)",
            "Median observed lifetime (days)",
            "Average orders per customer",
            "Average customer lifetime value",
            "Median customer lifetime value",
            "Average revenue per order",
        ],
        "value": [
            customer_kpis["CustomerNo"].nunique(),
            round(customer_kpis["customer_lifetime_days"].mean(), 2),
            round(customer_kpis["customer_lifetime_days"].median(), 2),
            round(customer_kpis["order_count"].mean(), 2),
            round(customer_kpis["total_revenue"].mean(), 2),
            round(customer_kpis["total_revenue"].median(), 2),
            round(customer_kpis["avg_revenue_per_order"].mean(), 2),
        ],
    }
)

customer_lifetime_by_country = (
    customer_kpis.groupby("Country", as_index=False)
    .agg(
        customers=("CustomerNo", "nunique"),
        avg_lifetime_days=("customer_lifetime_days", "mean"),
        median_lifetime_days=("customer_lifetime_days", "median"),
        avg_orders_per_customer=("order_count", "mean"),
        avg_customer_lifetime_value=("total_revenue", "mean"),
    )
)

for column in [
    "avg_lifetime_days",
    "median_lifetime_days",
    "avg_orders_per_customer",
    "avg_customer_lifetime_value",
]:
    customer_lifetime_by_country[column] = customer_lifetime_by_country[column].round(2)

display(customer_lifetime_summary)
display(customer_lifetime_by_country.sort_values("avg_customer_lifetime_value", ascending=False))

customer_kpis.head()

,metric,value
0,Customers,4463.00
1,Average observed lifetime (days),129.27
2,Median observed lifetime (days),94.00
3,Average orders per customer,3.89
4,Average customer lifetime value,6167.70
5,Median customer lifetime value,3046.70
6,Average revenue per order,1587.19


,Country,customers,avg_lifetime_days,median_lifetime_days,avg_orders_per_customer,avg_customer_lifetime_value
10,EIRE,13,167.54,196.0,16.92,46046.35
30,Singapore,1,262.00,262.0,4.00,28847.51
17,Iceland,1,365.00,365.0,7.00,22747.34
25,Norway,8,140.38,120.5,3.25,10410.27
14,Germany,89,158.74,121.0,4.88,10351.5
13,France,87,147.72,116.0,4.37,9627.77
3,Belgium,22,167.09,166.5,4.32,8971.41
24,Netherlands,8,166.38,203.0,6.62,8490.16
6,Channel Islands,5,180.00,201.0,3.00,8087.99
0,Australia,9,208.67,251.0,5.22,7835.77


,CustomerNo,Country,first_purchase,last_purchase,order_count,total_revenue,total_units,buyer_type,customer_lifetime_days,avg_revenue_per_order
0,12004,United Kingdom,2019-04-26,2019-04-26,1,1509.6,104,One-time buyer,0,1509.6
1,12006,United Kingdom,2019-05-05,2019-05-05,1,24.76,2,One-time buyer,0,24.76
2,12013,United Kingdom,2018-12-15,2018-12-15,1,69.96,3,One-time buyer,0,69.96
3,12024,United Kingdom,2019-06-16,2019-06-16,1,149.52,14,One-time buyer,0,149.52
4,12025,United Kingdom,2019-02-25,2019-02-25,1,1021.59,78,One-time buyer,0,1021.59
